# Axis 1 — Invariants (`20_invariants`)

**What this is.** Properties that must *always* hold, with no ground truth —
the inspectable form of `tests/test_invariants.py`. These catch the silent
structural failures: NaN-poisoned likelihood, frozen MH, non-reproducible
chains, crashes.

**How to read it.** Each cell prints the quantity, then asserts the rule.
You are not checking a value against a known answer here — you are checking
that a rule (finite, equal, non-zero, runs) holds.


## Setup


In [ ]:
import sys, numpy as np
sys.path.insert(0, '..'); sys.path.insert(0, '../..')
np.set_printoptions(precision=4, suppress=True)
from scipy.special import ndtr
import irtree.tree_spec as ts, irtree.data_prep as dp, irtree.priors as pr
import irtree.node_models as nm, irtree.mcmc_sampler as ms, irtree.likelihood as lk

def verdict(ok, label):
    print(f"  [{'PASS' if ok else 'FAIL'}]  {label}"); assert ok, label

def disengagement(P=200, I=12, seed=0, binary=True):
    rng = np.random.default_rng(seed)
    th1=rng.normal(0,1,P); th2=rng.normal(0,1,P)
    b1=rng.normal(0.5,0.7,I); b2=rng.normal(0,1,I)
    rg=rng.random((P,I))<ndtr(th1[:,None]-b1[None,:])
    co=rng.random((P,I))<ndtr(th2[:,None]-b2[None,:])
    Y=np.where(rg,1,np.where(co,3,2))
    spec=ts.disengagement_tree(binary=binary)
    return dp.expand_data(Y,spec,categories_start=1), spec


## 1. The `0.0 * nan` bug — likelihood must stay finite

`log_prob` returns NaN in irrelevant cells. The mask must *select*
(`np.where`), not *multiply* (`T * logprob`), because `0.0 * nan = nan`
poisons the whole sum. This made the log-likelihood trace entirely NaN and
froze every MH node.


In [ ]:
data, spec = disengagement(seed=1)
model = nm.ProbitRaschNode()
P, I = data.n_persons, data.n_items
for r in range(data.n_nodes):
    val = lk.log_node_lik(data.Y_star[r], data.T[r], np.zeros(P), np.zeros(I), model)
    print(f'  node {r} log-lik: {val:.2f}')
    verdict(np.isfinite(val), f'node {r} log-lik finite')

# and log_prob is NaN ONLY in irrelevant cells (never a relevant one)
lp = model.log_prob(data.Y_star[1], np.zeros(P), np.zeros(I))
rel = data.T[1]
print(f'  NaNs in relevant cells   : {int(np.sum(~np.isfinite(lp[rel])))}  (expect 0)')
print(f'  NaNs in irrelevant cells : {int(np.sum(~np.isfinite(lp[~rel])))}  (expected, harmless)')
verdict(np.all(np.isfinite(lp[rel])), 'log_prob finite in every relevant cell')


### The trace, end to end, must be finite


In [ ]:
data, spec = disengagement(seed=3)
models = [nm.ProbitRaschNode(), nm.ProbitRaschNode()]
priors = pr.build_default_priors(spec, models)
# Short chains: warmup_mode='fixed' (evidence mode needs n_iter >= 1600).
res = ms.run_chain(data, models, priors, n_iter=40, n_warmup=0, warmup_mode='fixed', seed=1, spec=spec)
lp = np.asarray(res.log_lik_trace, float)
print(f'  log-likelihood trace: {np.sum(~np.isfinite(lp))}/{lp.size} non-finite; range [{np.nanmin(lp):.1f}, {np.nanmax(lp):.1f}]')
verdict(np.all(np.isfinite(lp)), 'log-likelihood trace all finite')


## 2. MH liveness — a Logit node must accept > 0%

A pure-MH node stuck at 0% acceptance is a frozen chain producing numbers
that never move. This was caused by the NaN cache above; the invariant
guards it directly.


In [ ]:
data, spec = disengagement(P=300, I=15, seed=4)
models = [nm.ProbitRaschNode(), nm.LogitRaschNode()]
priors = pr.build_default_priors(spec, models)
res = ms.run_chain(data, models, priors, n_iter=500, n_warmup=300, warmup_mode='fixed', seed=1, spec=spec)
# a logit node's theta is updated person-wise (block theta_mh_<dim>) and its b
# item-wise (block b_mh_<group>)
acc = {k:v for k,v in res.accept_rates.items() if k.startswith(('theta_mh', 'b_mh'))}
print('  MH acceptance rates:', {k: round(v,4) for k,v in acc.items()})
verdict(len(acc) > 0, 'a Logit model records MH acceptance')
for k,v in acc.items(): verdict(v > 0.0, f'{k} accepts > 0%')


## 3. Cache correctness — total == sum, recompute is local


In [ ]:
data, spec = disengagement(seed=5)
models = [nm.ProbitRaschNode(), nm.ProbitRaschNode()]; R = data.n_nodes
theta = [np.zeros(data.n_persons) for _ in range(R)]
b = [np.zeros(data.n_items) for _ in range(R)]
cache = lk.init_cache(data, theta, b, models)
print(f'  cache.total = {cache.total:.3f}  sum = {float(np.sum(cache.node_logliks)):.3f}')
verdict(abs(cache.total - float(np.sum(cache.node_logliks))) < 1e-9, 'total == sum')
cache.node_logliks[0] += 3.5
verdict(abs(cache.total - float(np.sum(cache.node_logliks))) < 1e-9, 'total tracks after perturbation')

cache = lk.init_cache(data, theta, b, models); before = cache.node_logliks.copy()
lk.recompute_node(cache, 1, data, np.full(data.n_persons,0.7), b[1], models[1])
print(f'  node0 before/after: {before[0]:.3f} / {cache.node_logliks[0]:.3f}  (unchanged)')
print(f'  node1 before/after: {before[1]:.3f} / {cache.node_logliks[1]:.3f}  (updated)')
verdict(cache.node_logliks[0] == before[0], 'recompute(1) leaves node 0 untouched')
verdict(cache.node_logliks[1] != before[1], 'recompute(1) updates node 1')


## 4. Reproducibility — same seed identical, different seed different

The two-sided invariant: same seed → byte-identical (reproducible); *and*
different seed → different (a frozen RNG would pass the first alone).


In [ ]:
data, spec = disengagement(seed=7)
models = [nm.ProbitRaschNode(), nm.ProbitRaschNode()]
priors = pr.build_default_priors(spec, models)
# Short chains: warmup_mode='fixed' (evidence mode needs n_iter >= 1600).
a = ms.run_chain(data, models, priors, n_iter=60, n_warmup=0, warmup_mode='fixed', seed=42, spec=spec)
b_ = ms.run_chain(data, models, priors, n_iter=60, n_warmup=0, warmup_mode='fixed', seed=42, spec=spec)
c1 = ms.run_chain(data, models, priors, n_iter=60, n_warmup=0, warmup_mode='fixed', seed=1, spec=spec)
same = np.array_equal(np.asarray(a.chain), np.asarray(b_.chain))
diff = not np.array_equal(np.asarray(a.chain), np.asarray(c1.chain))
print(f'  seed 42 twice -> identical: {same}')
print(f'  seed 42 vs 1  -> different: {diff}')
verdict(same, 'same seed -> identical chain')
verdict(diff, 'different seed -> different chain (RNG threaded)')


## 5. GRM/GPCM run at all — the `int(NaN)` crash

Polytomous nodes index a probability array by the integer response;
`int(NaN)` becomes `INT_MIN` and crashes. The fix substitutes 0 for NaN in
the (irrelevant) cells. This invariant is 'does not raise' — the simplest
one, and the one that caught a regression lost from the delivered file.


In [ ]:
rng = np.random.default_rng(9); Pg, Ig = 200, 12
th1 = rng.normal(0,1,Pg); bg = rng.normal(0,1,Ig)
rg = rng.random((Pg,Ig)) < ndtr(th1[:,None]-bg[None,:])
u = rng.random((Pg,Ig)); cat = np.where(u<0.4,2,np.where(u<0.7,3,4))
Yg = np.where(rg,1,cat)
specg = ts.disengagement_tree(binary=False); datag = dp.expand_data(Yg,specg,categories_start=1)
modelsg = [nm.ProbitRaschNode(), nm.GRMNode(n_outcomes=3)]
priorsg = pr.build_default_priors(specg, modelsg)
resg = ms.run_chain(datag, modelsg, priorsg, n_iter=40, n_warmup=0, warmup_mode='fixed', seed=1, spec=specg)
lp = np.asarray(resg.log_lik_trace, float)
print(f'  GRM chain ran: shape {np.asarray(resg.chain).shape}, trace finite: {np.all(np.isfinite(lp))}')
verdict(np.asarray(resg.chain).shape[0]==40, 'GRM fit produced 40 stored iterations')
verdict(np.all(np.isfinite(lp)), 'GRM log-likelihood finite')


## Summary

Every always-true property holds: nothing is NaN or infinite, the MH node
moves, the cache is consistent, chains are reproducible-yet-seed-sensitive,
and GRM runs. These are the structural guards — the ones that catch a bug
*loudly* the moment a path executes, which is precisely what bitwise
regression cannot do.
